# Batch Processing Data Magnetotellurik (MT) Format EDI
Notebook ini berfungsi sebagai antarmuka utama untuk memproses beberapa file data MT berformat `.edi` secara otomatis. Semua logika pemrosesan kompleks disimpan dalam modul `edi_processing_tools.py` agar notebook ini tetap bersih dan fokus pada alur kerja analisis.

# 1. Import Packages & Initialization
Pertama, kita impor `pandas` untuk menampilkan hasil dan modul kustom kita, `edi_processing_tools`, yang berisi semua fungsi yang kita butuhkan.

In [9]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import edi_processing_tools_py2 as edt

data_folder = r"C:\Users\Lenovo X390\Downloads\trial_datashifa\trial_datashifa"

# 2. Read Site List
Tentukan nama file yang berisi daftar situs, lalu panggil fungsi `read_site_list` dari modul kita untuk mendapatkan daftar file yang akan diproses.


In [10]:
import os

edi_files = [f for f in os.listdir(data_folder) if f.endswith(".edi")]
print(edi_files)

site_list_path = os.path.join(data_folder, "..", "Sitelist.dat")
with open(site_list_path, "w") as f:
    for edi in sorted(edi_files):
        f.write(edi + "\n")   # atau tulis nama stasiunnya saja tanpa ".edi" kalau formatnya begitu

print("Sitelist.dat dibuat di:", os.path.abspath(site_list_path))

['MT1.edi', 'MT10.edi', 'MT2.edi', 'MT3.edi', 'MT4.edi', 'MT5.edi', 'MT6.edi', 'MT7.edi', 'MT8.edi', 'MT9.edi']
Sitelist.dat dibuat di: C:\Users\Lenovo X390\Downloads\trial_datashifa\Sitelist.dat


In [11]:
site_list_file = os.path.join(data_folder, "..", "Sitelist.dat")
result = edt.read_site_list(site_list_file)
if result and isinstance(result, tuple) and len(result) == 2:
    for site in result[1]:
        print(site)

MT1.edi
MT10.edi
MT2.edi
MT3.edi
MT4.edi
MT5.edi
MT6.edi
MT7.edi
MT8.edi
MT9.edi


In [12]:
import os
print("data_folder:", data_folder)
print("Isi folder edis:", os.listdir(data_folder))
print("Isi folder induk:", os.listdir(os.path.join(data_folder, "..")))

data_folder: C:\Users\Lenovo X390\Downloads\trial_datashifa\trial_datashifa
Isi folder edis: ['.ipynb_checkpoints', '1) PRE-PROCESSING.ipynb', '2)L-CURVE_rombak_v2(2).ipynb', '3) EXTRACT DEPTH.ipynb', '4) HORIZONTAL MAP.ipynb', '4__HORIZONTAL_MAP_FIXED (2).ipynb', '4__HORIZONTAL_MAP_FIXED.ipynb', '4__HORIZONTAL_MAP_FIXED_shif.ipynb', '5_VERTICAL_MAP_FIXED11.ipynb', 'AMT', 'CONVERT DMS-DECIMAL.ipynb', 'core_code2.py', 'DOP', 'edis', 'edi_processing_tools_py2.py', 'MT1.edi', 'MT10.edi', 'MT2.edi', 'MT3.edi', 'MT4.edi', 'MT5.edi', 'MT6.edi', 'MT7.edi', 'MT8.edi', 'MT9.edi', 'Sitelist.dat', 'SSQ', '__pycache__']
Isi folder induk: ['Sitelist.dat', 'trial_datashifa']


# 3. Parse EDI & Process SSQ Batch
Ini adalah inti dari notebook. Kita akan melakukan looping melalui setiap file, memprosesnya, dan mengumpulkan hasilnya. Di akhir, semua hasil akan digabungkan dan disimpan ke satu file CSV.

In [13]:
def fixed_parse_single_edi_file(file_path):
    try:
        with open(file_path, 'r', errors='ignore') as f: content = f.read()
    except FileNotFoundError: return None, None
    metadata = {'elevation': float(re.search(r'ELEV=(-?\d+\.?\d*)', content, re.IGNORECASE).group(1)) if re.search(r'ELEV=(-?\d+\.?\d*)', content, re.IGNORECASE) else 0.0}
    nfreq_match = re.search(r'NFREQ\s*=\s*(\d+)', content, re.IGNORECASE)
    if not nfreq_match: return None, None
    num_freqs = int(nfreq_match.group(1))
    cols = {"FREQ", "ZROT", "ZXXR", "ZXXI", "ZXX.VAR", "ZXYR", "ZXYI", "ZXY.VAR", "ZYXR", "ZYXI", "ZYX.VAR", "ZYYR", "ZYYI", "ZYY.VAR", "TROT", "TXR.EXP", "TXI.EXP", "TXVAR.EXP", "TYR.EXP", "TYI.EXP", "TYVAR.EXP"}
    data_dict = {name: [] for name in cols}
    for block in [b.strip() for b in content.split('>') if b.strip()]:
        first_word = block.split()[0] if block.split() else ""
        key = next((k for k in cols if first_word.startswith(k)), None)
        if key:
            data_dict[key].extend([float(num) for num in re.sub(r'//.*', '', ' '.join(block.splitlines()[1:])).split()])
    try:
        for key in cols:
            if not data_dict[key]: data_dict[key] = [0.0] * num_freqs
        return pd.DataFrame(data_dict), metadata
    except ValueError: return None, None

edt.parse_single_edi_file = fixed_parse_single_edi_file

raw_edi_files = [f for f in os.listdir(data_folder) if f.lower().endswith('.edi')]
all_sites_res = []
for edi_file in raw_edi_files:
    df, _ = edt.parse_single_edi_file(os.path.join(data_folder, edi_file))
    if df is None: continue
    period = 1.0 / (df['FREQ'] if 'FREQ' in df.columns else df['Freq (Hz)'])
    Zxx, Zxy, Zyx, Zyy = df['ZXXR']+1j*df['ZXXI'], df['ZXYR']+1j*df['ZXYI'], df['ZYXR']+1j*df['ZYXI'], df['ZYYR']+1j*df['ZYYI']
    det_Z = (Zxx * Zyy) - (Zxy * Zyx)
    df_with_zssq = edt.calculate_zssq_complex(df)
    res_df = edt.calculate_resistivity_and_phase(df_with_zssq)
    res_df['Rho_App_DET'] = 0.2 * period * np.abs(det_Z)
    res_df['Phase_DET'] = np.degrees(np.angle(det_Z)) / 2.0
    res_df['Zssq_Real'], res_df['Zssq_Imag'] = df_with_zssq['ZssqR'], df_with_zssq['ZssqI']
    res_df.insert(0, 'Station', os.path.splitext(edi_file)[0])
    all_sites_res.append(res_df)

if all_sites_res:
    os.makedirs(os.path.join(data_folder, 'SSQ'), exist_ok=True)
    pd.concat(all_sites_res, ignore_index=True).to_csv(os.path.join(data_folder, 'SSQ', 'SSQ ALL STATION.csv'), index=False)

# 4. Replace Tensor Components with ZSSQ
Sel ini akan membuat file .edi baru untuk setiap situs. File baru ini akan memiliki struktur yang sama dengan aslinya

In [14]:
def replace_components_with_zssq(orig_path, out_path, zssq_r, zssq_i, components=['ZXY', 'ZYX']):
    fmt = lambda v: [f"{' '.join(f'{x:.6f}' for x in v[i:i+5])}\n" for i in range(0, len(v), 5)] + ["\n"]
    r_lines, i_lines = fmt(zssq_r), fmt(zssq_i)
    with open(orig_path, 'r', errors='ignore') as infile, open(out_path, 'w') as outfile:
        skip = False
        for line in infile:
            upper = line.strip().upper()
            if any(upper.startswith(f'>{c}R') for c in components):
                outfile.write(line)
                outfile.writelines(r_lines)
                skip = True
            elif any(upper.startswith(f'>{c}I') for c in components):
                outfile.write(line)
                outfile.writelines(i_lines)
                skip = True
            elif upper.startswith('>'): skip = False
            if not skip: outfile.write(line)

csv_path = os.path.join(data_folder, 'SSQ', 'SSQ ALL STATION.csv')
if os.path.exists(csv_path):
    df_csv = pd.read_csv(csv_path)
    for site_name, group_df in df_csv.groupby('Station'):
        orig_path = os.path.join(data_folder, f"{site_name}.edi")
        if os.path.exists(orig_path):
            replace_components_with_zssq(orig_path, os.path.join(data_folder, 'SSQ', f"{site_name}_SSQ.edi"), group_df['Zssq_Real'].values, group_df['Zssq_Imag'].values)

# 5. AMT Frequency Filter

In [15]:
def trim_edi_to_amt(in_path, out_path, min_freq=0.1):
    try:
        with open(in_path, 'r', errors='ignore') as f: content = f.read()
    except FileNotFoundError: return False
    blocks = content.split('>')
    freqs = []
    for b in blocks:
        if b.strip().upper().startswith('FREQ'):
            parts = b.split('\n', 1)
            if len(parts) > 1: freqs = [float(x) for x in re.sub(r'//.*', '', parts[1]).split() if re.match(r'^-?\d+(\.\d+)?([eE][-+]?\d+)?$', x)]
            break
    if not freqs: return False
    mask = [f > min_freq for f in freqs]
    new_nfreq = sum(mask)
    if new_nfreq == 0: return False
    new_blocks = []
    for i, block in enumerate(blocks):
        if i == 0:
            new_blocks.append(block)
            continue
        parts = block.split('\n', 1)
        header, body = parts[0], parts[1] if len(parts) > 1 else ""
        first_word = header.split()[0].upper() if header.split() else ""
        header = re.sub(r'(NFREQ\s*=\s*)\d+', rf'\g<1>{new_nfreq}', header, flags=re.IGNORECASE)
        body = re.sub(r'(NFREQ\s*=\s*)\d+', rf'\g<1>{new_nfreq}', body, flags=re.IGNORECASE)
        if first_word in ['FREQ', 'ZROT', 'TROT'] or first_word.startswith(('ZXX', 'ZXY', 'ZYX', 'ZYY', 'TX', 'TY', 'RHO', 'PHS')):
            tokens = re.sub(r'//.*', '', body).split()
            if len(tokens) == len(freqs):
                ftokens = [v for v, m in zip(tokens, mask) if m]
                new_blocks.append(f"{header}\n{chr(10).join([' '.join(ftokens[j:j+5]) for j in range(0, len(ftokens), 5)])}\n")
                continue
        new_blocks.append(f"{header}{chr(10) + body if len(parts) > 1 else ''}")
    with open(out_path, 'w') as f: f.write('>'.join(new_blocks))
    return True

output_folder = os.path.join(data_folder, "AMT", "00. DATA EDI")
os.makedirs(output_folder, exist_ok=True)

for file in [f for f in os.listdir(os.path.join(data_folder, "SSQ")) if f.lower().endswith('.edi')]:
    trim_edi_to_amt(os.path.join(data_folder, "SSQ", file), os.path.join(output_folder, f"{file[:-4]}_AMT.edi"))

# Ubah nilai min_freq dari 0.1 menjadi 1.0 (artinya memotong periode > 1 detik)
for file in [f for f in os.listdir(os.path.join(data_folder, "SSQ")) if f.lower().endswith('.edi')]:
    trim_edi_to_amt(
        os.path.join(data_folder, "SSQ", file), 
        os.path.join(output_folder, f"{file[:-4]}_AMT.edi"),
        min_freq=1.0
    )

# 6. Calculate Apparent Resistivity & Phase (Post-SSQ & AMT)

In [16]:
amt_folder = os.path.join(data_folder, 'AMT', '00. DATA EDI')

all_sites_data = []
for fpath in [os.path.join(amt_folder, f) for f in os.listdir(amt_folder) if f.lower().endswith('.edi')]:
    df, _ = edt.parse_single_edi_file(fpath)
    if df is None: continue
    res_df = edt.calculate_resistivity_and_phase(df)
    f_col = 'FREQ' if 'FREQ' in df.columns else 'Freq(Hz)'
    if f_col in df.columns:
        det_Z = ((df['ZXXR']+1j*df['ZXXI']) * (df['ZYYR']+1j*df['ZYYI'])) - ((df['ZXYR']+1j*df['ZXYI']) * (df['ZYXR']+1j*df['ZYXI']))
        res_df['Rho_App_DET'], res_df['Phase_DET'] = 0.2 * (1.0/df[f_col]) * np.abs(det_Z), np.degrees(np.angle(det_Z)) / 2.0
    res_df.insert(0, 'Station', os.path.splitext(os.path.basename(fpath))[0])
    if 'FREQ' in res_df.columns: res_df.rename(columns={'FREQ': 'Freq(Hz)'}, inplace=True)
    if 'Period(s)' not in res_df.columns and 'Freq(Hz)' in res_df.columns: res_df.insert(2, 'Period(s)', 1.0 / res_df['Freq(Hz)'])
    all_sites_data.append(res_df)

print("Jumlah data terkumpul:", len(all_sites_data))

if all_sites_data:
    os.makedirs(os.path.join(data_folder, 'DOP'), exist_ok=True)
    pd.concat(all_sites_data, ignore_index=True).replace([np.inf, -np.inf], np.nan).to_csv(os.path.join(data_folder, 'DOP', 'APPRES & PHASE AFTER SSQ & AMT ALL STATION.csv'), index=False)
    print("CSV berhasil dibuat.")
else:
    print("Masih kosong — cek parse_single_edi_file, mungkin ada file yang gagal diparsing.")

Jumlah data terkumpul: 10
CSV berhasil dibuat.


# 7. Depth of Penetration DOP & Plotting

In [17]:
DENOM = 2 * np.pi * (4 * np.pi * 1e-7)

def calc_dop(df):
    p_col = 'Period(s)' if 'Period(s)' in df.columns else ('Period (s)' if 'Period (s)' in df.columns else None)
    if not p_col:
        df['Period(s)'] = 1.0 / df['Freq(Hz)' if 'Freq(Hz)' in df.columns else 'FREQ']
        p_col = 'Period(s)'
    pv = np.where(df[p_col] > 0, df[p_col], np.nan)
    
    # Hitung DOP dengan pembersihan data anomali/ekstrem
    for c, name in [('Rho_App_ZXY', 'DOP_XY_m'), ('Rho_App_ZYX', 'DOP_YX_m'), ('Rho_App_DET', 'DOP_DET_m')]:
        if c in df.columns:
            # Saring hanya nilai resistivitas rasional (0 < rho < 1,000,000)
            valid_rho = np.where((df[c] > 0) & (df[c] < 1e6), df[c], np.nan)
            dop_val = np.sqrt((valid_rho * pv) / DENOM)
            
            # Buang nilai kedalaman yang tak masuk akal (> 100 km / 100,000 m)
            df[name] = np.where(dop_val < 100000, dop_val, np.nan)
            
    return df

def plot_dop(df_raw, df_amt, site, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 7), facecolor='#FFFDF6')
    for ax, df, suf in zip(axes, [df_raw, df_amt], ["", "_SSQ"]):
        ax.set_facecolor('white')
        [spine.set(edgecolor='black', linewidth=1.5) for spine in ax.spines.values()]
        if df is None or df.empty:
            ax.text(0.5, 0.5, "Data Not Available", ha='center', va='center')
            ax.set_title(f"Depth of Investigation {site}{suf}")
            continue
        df = df.sort_values('Period(s)')
        for col, clr, lbl, mk in [('DOP_XY_m','#3B5998','TE','o'), ('DOP_YX_m','#C0392B','TM','o'), ('DOP_DET_m','#27AE60','DET','s')]:
            if col in df.columns and not df[col].isna().all():
                ax.plot(df[col]/1000, df['Period(s)'], marker=mk, ms=5 if mk=='o' else 4, ls='-', lw=1.5, color=clr, label=lbl)
        ax.set(xscale='log', yscale='log', xlabel='Depth (km)', ylabel='Period (s)', title=f"Depth of Investigation {site}{suf}")
        ax.invert_yaxis()
        ax.grid(True, which='both', color='gray', ls='--', lw=0.5, alpha=0.5)
        ax.legend(loc='upper right').get_frame().set_alpha(1)
        ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=12))
    plt.tight_layout(pad=3.0)
    plt.savefig(os.path.join(out_dir, f"{site}_DOP.png"), dpi=300, bbox_inches='tight')
    plt.close()

raw_csv, amt_csv, dop_dir = os.path.join(data_folder, "SSQ", "SSQ ALL STATION.csv"), os.path.join(data_folder, "DOP", "APPRES & PHASE AFTER SSQ & AMT ALL STATION.csv"), os.path.join(data_folder, "DOP")
if os.path.exists(raw_csv) and os.path.exists(amt_csv):
    df_raw = calc_dop(pd.read_csv(raw_csv))
    df_ssq = calc_dop(pd.read_csv(amt_csv)[pd.read_csv(amt_csv)['Station'].astype(str).str.contains('_SSQ')].copy())
    df_raw['Station'], df_ssq['Station'] = df_raw['Station'].astype(str).str.replace(r'_SSQ.*', '', regex=True), df_ssq['Station'].astype(str).str.replace(r'_SSQ.*', '', regex=True)
    
    for site in set(df_raw['Station']).union(set(df_ssq['Station'])):
        plot_dop(df_raw[df_raw['Station'] == site], df_ssq[df_ssq['Station'] == site], site, dop_dir)
        
    dop_cols = ['DOP_XY_m', 'DOP_YX_m', 'DOP_DET_m']
    m_raw = df_raw.groupby('Station')[[c for c in dop_cols if c in df_raw.columns]].max().add_suffix('_RAW (m)').reset_index()
    m_ssq = df_ssq.groupby('Station')[[c for c in dop_cols if c in df_ssq.columns]].max().add_suffix('_SSQ (m)').reset_index()
    pd.merge(m_raw, m_ssq, on='Station', how='outer').round(1).to_csv(os.path.join(dop_dir, "DOP ALL STATION.csv"), index=False, na_rep='N/A')

# 8. (Optional) 8. Export Ogawa-Uchida Format
Sel ini akan mengambil data yang telah diproses dan menyimpannya ke dalam format file tunggal yang sesuai untuk perangkat lunak inversi 2D. Anda dapat menentukan nama file output dan mode yang diinginkan (`TM` atau `TE`).

In [50]:
if 'all_sites_data' in locals() and all_sites_data:
    edt.write_ogawa_format(all_sites_data, 'TE_InputInversi2D.dat', mode='TE')